# Bundesliga Team Season Statistics Extractor

This notebook retrieves team-level overall season statistics for every Bundesliga team listed in **bundesliga_teams.json**. It requests:

    https://www.sofascore.com/api/v1/team/{team_id}/unique-tournament/35/season/{season_id}/statistics/overall

## Requirements and portable execution

- Place **bundesliga_teams.json** in the configured project output directory as this notebook.
- Start the Jupyter or VS Code kernel with that directory as its working directory. All local paths derive from the working directory; there are no machine-specific paths.
- Install the required packages if needed with: **%pip install pandas beautifulsoup4 undetected-chromedriver**.
- Chrome starts with **uc.Chrome()**. One browser session is reused for every request and closed in a finally block.

## Matchday and season selection

The notebook accepts matchdays 1–34. Matchday 1 uses the previous season (77333), matchdays 2–5 use the previous and current seasons (77333 and 97464), and matchday 6 onward uses only the current season (97464).

## Outputs

The notebook writes matching **bundesliga_team_statistics_md_{matchday}_{timestamp}.json** and **.csv** files in the configured project folder. JSON preserves nested statistics. CSV exposes all statistic keys dynamically. Failed team-season requests are recorded without stopping later requests.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    SOFASCORE_REFERENCE_DIR,
    SOFASCORE_TEAM_STATS_DIR,
    ensure_directory,
)


## 1. Imports

Load standard-library utilities plus pandas, Beautiful Soup, and undetected Chrome. Missing third-party packages produce an installation hint.


In [2]:
# Import the libraries required by this notebook step.
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
except ImportError as exc:
    raise ImportError(
        "Required packages are missing from this notebook kernel. Install them with: "
        "%pip install pandas beautifulsoup4 undetected-chromedriver"
    ) from exc


## 2. Configuration

Keep season IDs, tournament details, delay bounds, filenames, removable metadata fields, and identifier columns in one place.


In [3]:
# Set workflow configuration value: PREVIOUS_SEASON_ID.
PREVIOUS_SEASON_ID = 77333
# Set workflow configuration value: CURRENT_SEASON_ID.
CURRENT_SEASON_ID = 97464
# Set workflow configuration value: TOURNAMENT_ID.
TOURNAMENT_ID = 35
# Set workflow configuration value: CHROME_MAJOR_VERSION.
CHROME_MAJOR_VERSION = None  # Let undetected-chromedriver auto-detect Chrome.
# Set workflow configuration value: MIN_MATCHDAY.
MIN_MATCHDAY = 1
# Set workflow configuration value: MAX_MATCHDAY.
MAX_MATCHDAY = 34
# Set workflow configuration value: MIN_DELAY_SECONDS.
MIN_DELAY_SECONDS = 0.5
# Set workflow configuration value: MAX_DELAY_SECONDS.
MAX_DELAY_SECONDS = 1.5
# Set workflow configuration value: INPUT_FILENAME.
INPUT_FILENAME = "bundesliga_teams.json"
# Set workflow configuration value: NOTEBOOK_FILENAME.
NOTEBOOK_FILENAME = "02_sofascore_team_stats.ipynb"

# Set workflow configuration value: ENDPOINT_TEMPLATE.
ENDPOINT_TEMPLATE = (
    "https://www.sofascore.com/api/v1/team/{team_id}/"
    "unique-tournament/{tournament_id}/season/{season_id}/statistics/overall"
)
# Set workflow configuration value: SEASON_LABELS.
SEASON_LABELS = {
    PREVIOUS_SEASON_ID: "previous_season",
    CURRENT_SEASON_ID: "current_season",
}
# Set workflow configuration value: REMOVABLE_STATISTICS_FIELDS.
REMOVABLE_STATISTICS_FIELDS = ("id", "statisticsType", "awardedMatches")
# Set workflow configuration value: IDENTIFIER_COLUMNS.
IDENTIFIER_COLUMNS = [
    "team",
    "team_id",
    "season_label",
    "season_id",
    "matchday_at_capture",
    "capture_timestamp",
]


## 3. Resolve local paths

Use the resolved working directory as the portable base directory. After moving the notebook, start its kernel from the destination directory and place **bundesliga_teams.json** there. No parent search or absolute fallback is used.


In [4]:
base_directory = ensure_directory(SOFASCORE_TEAM_STATS_DIR)
teams_path = SOFASCORE_REFERENCE_DIR / INPUT_FILENAME

# Validate the input before continuing with later processing.
if not teams_path.is_file():
    raise FileNotFoundError(
        f"Required input file not found: {teams_path}. Run the team-reference notebook first."
    )

print(f"Output directory: {base_directory}")
print(f"Team input file: {teams_path}")


Output directory: C:\kickbase project\outputs\sofascore\team_stats
Team input file: C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json


## 4. Load and validate Bundesliga teams

Read the UTF-8 team mapping, verify its IDs and names, and normalize it into an ordered list without hard-coding the team count.


In [5]:
# Handle expected failures with a clear, actionable message.
try:
    teams_text = teams_path.read_text(encoding="utf-8")
except UnicodeDecodeError as exc:
    raise ValueError(f"{teams_path} is not valid UTF-8: {exc}") from exc
except OSError as exc:
    raise OSError(f"Could not read {teams_path}: {exc}") from exc

# Handle expected failures with a clear, actionable message.
try:
    raw_teams = json.loads(teams_text)
except json.JSONDecodeError as exc:
    raise ValueError(
        f"{teams_path} does not contain valid JSON "
        f"(line {exc.lineno}, column {exc.colno})."
    ) from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_teams, dict) or not raw_teams:
    raise ValueError("The team file must contain a non-empty JSON object.")

teams: list[dict[str, Any]] = []
seen_team_ids: set[int] = set()

# Process each available item while preserving the current workflow state.
for entry_number, (raw_team_id, team_record) in enumerate(raw_teams.items(), start=1):
    # Handle expected failures with a clear, actionable message.
    try:
        key_team_id = int(raw_team_id)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Team entry {entry_number} has a non-integer object key: {raw_team_id!r}."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(team_record, dict):
        raise ValueError(f"Team entry {raw_team_id!r} must be a JSON object.")

    team_id = team_record.get("team_id")
    team_name = team_record.get("team")
    # Validate the input before continuing with later processing.
    if not isinstance(team_id, int) or isinstance(team_id, bool) or team_id <= 0:
        raise ValueError(
            f"Team entry {raw_team_id!r} must contain a positive integer team_id."
        )
    # Validate the input before continuing with later processing.
    if key_team_id != team_id:
        raise ValueError(
            f"Team object key {raw_team_id!r} does not match team_id {team_id}."
        )
    # Validate the input before continuing with later processing.
    if not isinstance(team_name, str) or not team_name.strip():
        raise ValueError(
            f"Team entry {raw_team_id!r} must contain a non-empty team name."
        )
    # Validate the input before continuing with later processing.
    if team_id in seen_team_ids:
        raise ValueError(f"Duplicate team_id found: {team_id}.")

    seen_team_ids.add(team_id)
    teams.append({"team_id": team_id, "team": team_name.strip()})

print(f"Validated {len(teams)} team(s).")


Validated 18 team(s).


## 5. Ask for the matchday and select seasons

Re-prompt until an integer from 1 through 34 is entered, apply the fixed season rules, and calculate the request total dynamically.


In [6]:
# Handle for matchday for reuse in the workflow.
def prompt_for_matchday() -> int:
    while True:
        raw_value = input("Enter the current Bundesliga matchday: ").strip()
        # Handle expected failures with a clear, actionable message.
        try:
            value = int(raw_value)
        except ValueError:
            print("Invalid input. Enter an integer from 1 through 34.")
            continue

        if MIN_MATCHDAY <= value <= MAX_MATCHDAY:
            return value
        print("Invalid matchday. Enter an integer from 1 through 34.")


# Select season IDs for reuse in the workflow.
def select_season_ids(selected_matchday: int) -> list[int]:
    if selected_matchday == 1:
        return [PREVIOUS_SEASON_ID]
    if 2 <= selected_matchday <= 5:
        return [PREVIOUS_SEASON_ID, CURRENT_SEASON_ID]
    return [CURRENT_SEASON_ID]


matchday = prompt_for_matchday()
season_ids = select_season_ids(matchday)
total_requests = len(teams) * len(season_ids)
selected_season_text = ", ".join(
    f"{SEASON_LABELS[season_id]} ({season_id})" for season_id in season_ids
)

print(f"Matchday: {matchday}")
print(f"Selected seasons: {selected_season_text}")
print(f"Teams: {len(teams)}")
print(f"Total requests: {total_requests}")


Enter the current Bundesliga matchday:  2


Matchday: 2
Selected seasons: previous_season (77333), current_season (97464)
Teams: 18
Total requests: 36


## 6. Build URLs and validate SofaScore responses

Build team-season URLs, extract JSON from Chrome's preformatted response wrapper, validate the statistics object, and remove only approved metadata fields.


In [7]:
# Define Sofa Score Statistics Error to keep related behaviour explicit.
class SofaScoreStatisticsError(RuntimeError):
    """Raised when a SofaScore team-statistics response is unusable."""


# Build statistics url for reuse in the workflow.
def build_statistics_url(team_id: int, season_id: int) -> str:
    return ENDPOINT_TEMPLATE.format(
        team_id=team_id,
        tournament_id=TOURNAMENT_ID,
        season_id=season_id,
    )


# Parse and validate statistics page for reuse in the workflow.
def parse_statistics_page(page_source: str, url: str) -> dict[str, Any]:
    # Validate the input before continuing with later processing.
    if not isinstance(page_source, str) or not page_source.strip():
        raise SofaScoreStatisticsError(f"SofaScore returned an empty page for {url}.")

    soup = BeautifulSoup(page_source, "html.parser")
    pre_tag = soup.find("pre")
    # Validate the input before continuing with later processing.
    if pre_tag is None:
        raise SofaScoreStatisticsError(
            f"SofaScore returned unexpected HTML without a pre response for {url}."
        )

    response_text = pre_tag.get_text().strip()
    # Validate the input before continuing with later processing.
    if not response_text:
        raise SofaScoreStatisticsError(f"SofaScore returned an empty response for {url}.")

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise SofaScoreStatisticsError(
            f"SofaScore did not return valid JSON for {url} "
            f"(line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise SofaScoreStatisticsError(f"SofaScore response for {url} is not a JSON object.")
    # Validate the input before continuing with later processing.
    if "statistics" not in payload:
        raise SofaScoreStatisticsError(
            f"SofaScore response for {url} has no 'statistics' field."
        )

    statistics = payload["statistics"]
    # Validate the input before continuing with later processing.
    if not isinstance(statistics, dict):
        raise SofaScoreStatisticsError(
            f"SofaScore 'statistics' field for {url} is not a JSON object."
        )
    # Validate the input before continuing with later processing.
    if not statistics:
        raise SofaScoreStatisticsError(
            f"SofaScore returned an empty statistics object for {url}."
        )

    cleaned_statistics = statistics.copy()
    # Process each available item while preserving the current workflow state.
    for field_name in REMOVABLE_STATISTICS_FIELDS:
        cleaned_statistics.pop(field_name, None)
    # Validate the input before continuing with later processing.
    if not cleaned_statistics:
        raise SofaScoreStatisticsError(
            f"SofaScore returned no meaningful statistics for {url}."
        )
    return cleaned_statistics


## 7. Retrieve statistics with one reusable Chrome session

Open a detected-version Chrome session once, process every team-season combination independently, record failures, pause randomly between requests, and close the browser safely.


In [8]:
successful_records: list[dict[str, Any]] = []
failed_requests: list[dict[str, Any]] = []
driver = None
request_index = 0
run_started_at = datetime.now(timezone.utc)

# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome()

    # Process each available item while preserving the current workflow state.
    for team_record in teams:
        team_id = team_record["team_id"]
        team_name = team_record["team"]

        # Process each available item while preserving the current workflow state.
        for season_id in season_ids:
            request_index += 1
            season_label = SEASON_LABELS[season_id]
            url = build_statistics_url(team_id, season_id)
            print(f"[{request_index}/{total_requests}] {team_name} — {season_label}")

            # Handle expected failures with a clear, actionable message.
            try:
                driver.get(url)
                statistics = parse_statistics_page(driver.page_source, url)
                capture_timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
                successful_records.append(
                    {
                        "team": team_name,
                        "team_id": team_id,
                        "season_label": season_label,
                        "season_id": season_id,
                        "matchday_at_capture": matchday,
                        "capture_timestamp": capture_timestamp,
                        "statistics": statistics,
                    }
                )
            except Exception as exc:
                error_message = str(exc).strip() or repr(exc)
                failed_requests.append(
                    {
                        "team_id": team_id,
                        "team": team_name,
                        "season_id": season_id,
                        "season_label": season_label,
                        "url": url,
                        "error_type": type(exc).__name__,
                        "error": error_message,
                    }
                )
                print(f"  Failed: {type(exc).__name__}: {error_message}")

            if request_index < total_requests:
                delay_seconds = random.uniform(MIN_DELAY_SECONDS, MAX_DELAY_SECONDS)
                time.sleep(delay_seconds)
finally:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
            print("Chrome driver closed.")
        except Exception as shutdown_error:
            print(
                f"Chrome driver shutdown warning: "
                f"{type(shutdown_error).__name__}: {shutdown_error}"
            )
        finally:
            driver = None


[1/36] FC Bayern München — previous_season
[2/36] FC Bayern München — current_season
[3/36] VfB Stuttgart — previous_season
[4/36] VfB Stuttgart — current_season
[5/36] 1. FC Köln — previous_season
[6/36] 1. FC Köln — current_season
[7/36] TSG Hoffenheim — previous_season
[8/36] TSG Hoffenheim — current_season
[9/36] 1. FC Union Berlin — previous_season
[10/36] 1. FC Union Berlin — current_season
[11/36] Eintracht Frankfurt — previous_season
[12/36] Eintracht Frankfurt — current_season
[13/36] 1. FSV Mainz 05 — previous_season
[14/36] 1. FSV Mainz 05 — current_season
[15/36] SC Paderborn 07 — previous_season
  Failed: SofaScoreStatisticsError: SofaScore response for https://www.sofascore.com/api/v1/team/2561/unique-tournament/35/season/77333/statistics/overall has no 'statistics' field.
[16/36] SC Paderborn 07 — current_season
[17/36] RB Leipzig — previous_season
[18/36] RB Leipzig — current_season
[19/36] Borussia M'gladbach — previous_season
[20/36] Borussia M'gladbach — current_seas

## 8. Build the dynamic DataFrame

Create one row per successful request, place identifiers first, discover statistic columns dynamically, avoid name collisions, and encode nested values as compact JSON for CSV.


In [9]:
statistic_keys = sorted(
    {
        statistic_key
        for record in successful_records
        for statistic_key in record["statistics"]
    }
)

used_column_names = set(IDENTIFIER_COLUMNS)
csv_statistic_columns: dict[str, str] = {}
# Process each available item while preserving the current workflow state.
for statistic_key in statistic_keys:
    column_name = statistic_key
    if column_name in used_column_names:
        column_name = f"stat__{statistic_key}"
    while column_name in used_column_names:
        column_name = f"_{column_name}"
    csv_statistic_columns[statistic_key] = column_name
    used_column_names.add(column_name)

dataframe_rows: list[dict[str, Any]] = []
# Process each available item while preserving the current workflow state.
for record in successful_records:
    row = {column_name: record[column_name] for column_name in IDENTIFIER_COLUMNS}
    # Process each available item while preserving the current workflow state.
    for statistic_key, column_name in csv_statistic_columns.items():
        value = record["statistics"].get(statistic_key)
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False, separators=(",", ":"))
        row[column_name] = value
    dataframe_rows.append(row)

dataframe_columns = IDENTIFIER_COLUMNS + list(csv_statistic_columns.values())
team_statistics_df = pd.DataFrame(dataframe_rows, columns=dataframe_columns)
print(
    f"Built DataFrame with {len(team_statistics_df)} row(s) and "
    f"{len(team_statistics_df.columns)} column(s)."
)


Built DataFrame with 33 row(s) and 128 column(s).


## 9. Validate and save JSON and CSV outputs

Build a two-key JSON document, verify internal invariants, and write matching UTF-8 JSON and CSV files with one shared timestamp stem.


In [10]:
run_finished_at = datetime.now(timezone.utc)
success_count = len(successful_records)
failure_count = len(failed_requests)

# Validate the input before continuing with later processing.
if success_count + failure_count != total_requests:
    raise RuntimeError(
        "Internal request-count mismatch: successes plus failures do not equal "
        "the planned request total."
    )

valid_team_ids = {team_record["team_id"] for team_record in teams}
valid_season_ids = set(season_ids)
# Process each available item while preserving the current workflow state.
for record_number, record in enumerate(successful_records, start=1):
    # Validate the input before continuing with later processing.
    if record["team_id"] not in valid_team_ids:
        raise RuntimeError(f"Successful record {record_number} has an unknown team_id.")
    # Validate the input before continuing with later processing.
    if record["season_id"] not in valid_season_ids:
        raise RuntimeError(f"Successful record {record_number} has an unselected season_id.")
    # Validate the input before continuing with later processing.
    if not isinstance(record["statistics"], dict) or not record["statistics"]:
        raise RuntimeError(
            f"Successful record {record_number} has no validated statistics."
        )

filename_timestamp = run_started_at.strftime("%Y%m%dT%H%M%SZ")
output_stem = f"bundesliga_team_statistics_md_{matchday}_{filename_timestamp}"
json_output_path = base_directory / f"{output_stem}.json"
csv_output_path = base_directory / f"{output_stem}.csv"

output_data = {
    "metadata": {
        "competition": "Bundesliga",
        "tournament_id": TOURNAMENT_ID,
        "endpoint_template": ENDPOINT_TEMPLATE,
        "source_team_file": INPUT_FILENAME,
        "matchday": matchday,
        "selected_seasons": [
            {
                "season_id": season_id,
                "season_label": SEASON_LABELS[season_id],
            }
            for season_id in season_ids
        ],
        "capture_started_at": run_started_at.isoformat(timespec="seconds"),
        "capture_finished_at": run_finished_at.isoformat(timespec="seconds"),
        "team_count": len(teams),
        "total_requests": total_requests,
        "success_count": success_count,
        "failure_count": failure_count,
        "removed_statistics_fields": list(REMOVABLE_STATISTICS_FIELDS),
        "failed_requests": failed_requests,
    },
    "teams": successful_records,
}

# Handle expected failures with a clear, actionable message.
try:
    json_output_path.write_text(
        json.dumps(output_data, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
except OSError as exc:
    raise OSError(f"Could not write JSON output {json_output_path}: {exc}") from exc

# Handle expected failures with a clear, actionable message.
try:
    team_statistics_df.to_csv(
        csv_output_path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
    )
except OSError as exc:
    raise OSError(f"Could not write CSV output {csv_output_path}: {exc}") from exc


## 10. Print the execution summary

Report the selected scope, request results, output paths, any failures, and a small table preview.


In [11]:
# Run this step and display the result for inspection.
print("\nExtraction complete.")
print(f"Matchday: {matchday}")
print(f"Selected seasons: {selected_season_text}")
print(f"Teams: {len(teams)}")
print(f"Total requests: {total_requests}")
print(f"Successful requests: {success_count}")
print(f"Failed requests: {failure_count}")
print(f"JSON output: {json_output_path.resolve()}")
print(f"CSV output: {csv_output_path.resolve()}")

if failed_requests:
    failure_columns = [
        "team", "team_id", "season_label", "season_id", "error_type", "error"
    ]
    failure_df = pd.DataFrame(failed_requests, columns=failure_columns)
    print("\nFailed request summary:")
    print(failure_df.to_string(index=False))

team_statistics_df.head()



Extraction complete.
Matchday: 2
Selected seasons: previous_season (77333), current_season (97464)
Teams: 18
Total requests: 36
Successful requests: 33
Failed requests: 3
JSON output: C:\kickbase project\outputs\sofascore\team_stats\bundesliga_team_statistics_md_2_20260904T084406Z.json
CSV output: C:\kickbase project\outputs\sofascore\team_stats\bundesliga_team_statistics_md_2_20260904T084406Z.csv

Failed request summary:
            team  team_id    season_label  season_id               error_type                                                                                                                                             error
 SC Paderborn 07     2561 previous_season      77333 SofaScoreStatisticsError SofaScore response for https://www.sofascore.com/api/v1/team/2561/unique-tournament/35/season/77333/statistics/overall has no 'statistics' field.
SV 07 Elversberg     2598 previous_season      77333 SofaScoreStatisticsError SofaScore response for https://www.sofascore.co

,team,team_id,season_label,season_id,matchday_at_capture,capture_timestamp,accurateCrosses,accurateCrossesPercentage,accurateFinalThirdPassesAgainst,accurateLongBalls,...,totalGroundDuels,totalLongBalls,totalOppositionHalfPasses,totalOwnHalfPasses,totalPasses,totalPassesAgainst,touches,yellowCards,yellowCardsAgainst,yellowRedCards
0,FC Bayern München,2672,previous_season,77333,2,2026-09-04T08:44:10+00:00,115,21.217712,1755,721,...,2192,1107,14910,8920,23830,11227,NaN,57,51,1
1,FC Bayern München,2672,current_season,97464,2,2026-09-04T08:44:12+00:00,2,20.000000,78,15,...,64,29,317,231,548,370,740.0,1,3,0
2,VfB Stuttgart,2677,previous_season,77333,2,2026-09-04T08:44:13+00:00,177,25.991189,2364,494,...,1990,1030,10294,7292,17586,12866,NaN,63,53,0
3,VfB Stuttgart,2677,current_season,97464,2,2026-09-04T08:44:14+00:00,2,13.333333,145,5,...,64,13,222,148,370,548,552.0,3,1,0
4,1. FC Köln,2671,previous_season,77333,2,2026-09-04T08:44:14+00:00,175,27.301092,3130,439,...,1940,1075,7198,6686,13884,15750,NaN,63,73,1


In [12]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
